# CC5 scraper: how jobs in Great Britain have changed since the internet

This notebook **scrapes** a web table of jobs by industry at every census from 1841 to 2021, cleans it, reshapes it into **tidy (long) format**, saves it as a CSV, and builds a chart.

- **Source:** A Vision of Britain through Time (University of Portsmouth), *Great Britain: Standard Industrial Classification (2007)*: https://www.visionofbritain.org.uk/unit/10090283/cube/INDUSTRY_GEN_2021
- **Required acknowledgement:** "This work is based on data provided through www.VisionofBritain.org.uk and uses historical material which is copyright of the Great Britain Historical GIS Project and the University of Portsmouth." On a web page, "www.VisionofBritain.org.uk" must be a working link.
- **Outputs:** `uk_jobs_by_industry_tidy.csv` (tidy data) and `cc5_scraper_chart.json` (Vega-Lite chart).

In [ ]:
import io
import re

import altair as alt
import pandas as pd
import requests

## 1. Download the web page

In [ ]:
URL = "https://www.visionofbritain.org.uk/unit/10090283/cube/INDUSTRY_GEN_2021"
HEADERS = {"User-Agent": "PP434 student project (course exercise)"}

response = requests.get(URL, headers=HEADERS, timeout=60)
response.raise_for_status()
html = response.text
print(f"Downloaded {len(html):,} characters from {URL}")

## 2. Scrape the table

`pandas.read_html` finds every `<table>` on the page. The one we want has a **Year** column followed by one column per industry (A: Agriculture … R, S, T, U: Other services).

In [ ]:
tables = pd.read_html(io.StringIO(html))
print(f"Found {len(tables)} tables on the page")

# Pick the table whose first column is Year and which has the industry columns
def is_industry_table(t):
    cols = [str(c) for c in t.columns]
    return any("year" in c.lower() for c in cols[:1]) and any("Manufacturing" in c for c in cols)

raw = next(t for t in tables if is_industry_table(t))
print(raw.shape)
raw.head(3)

## 3. Clean

Problems in the scraped table, and the fix for each:

- Every number cell also contains the link text **"Show data context"** → removed.
- Numbers have **thousands commas** ("1,457,404") → removed, then converted to numbers.
- Column names combine a letter code and a name ("C: Manufacturing") → split into `code` and `industry`.
- **1951 has 0** for some industries because that census grouped them differently → kept, but flagged as "not comparable" rather than treated as real zero jobs.

In [ ]:
df = raw.copy()
df.columns = [str(col).strip() for col in df.columns]
year_col = df.columns[0]

def to_number(cell):
    """'1,457,404 Show data context' -> 1457404"""
    text = re.sub(r"show data context", "", str(cell), flags=re.I)
    text = text.replace(",", "").strip()
    return pd.to_numeric(text, errors="coerce")

df[year_col] = pd.to_numeric(df[year_col], errors="coerce")
df = df.dropna(subset=[year_col])
for col in df.columns[1:]:
    df[col] = df[col].map(to_number)
df = df.rename(columns={year_col: "year"})
df["year"] = df["year"].astype(int)
df.head(3)

## 4. Reshape into tidy (long) format: one row per year × industry

In [ ]:
tidy = df.melt(id_vars="year", var_name="column", value_name="jobs")

# "C: Manufacturing" -> code "C", industry "Manufacturing"
split = tidy["column"].str.split(":", n=1, expand=True)
tidy["code"] = split[0].str.strip()
tidy["industry"] = split[1].str.strip().str.rstrip(".")
tidy = tidy.drop(columns="column")

tidy["share_pct"] = tidy["jobs"] / tidy.groupby("year")["jobs"].transform("sum") * 100
tidy["comparable"] = ~((tidy["year"] == 1951) & (tidy["jobs"] == 0))   # 1951 classification gaps
tidy = tidy[["year", "code", "industry", "jobs", "share_pct", "comparable"]].sort_values(["year", "code"])

tidy.to_csv("uk_jobs_by_industry_tidy.csv", index=False)
print(f"Saved uk_jobs_by_industry_tidy.csv: {len(tidy)} rows, {tidy['year'].nunique()} census years, {tidy['industry'].nunique()} industries")
tidy.head(8)

## 5. Chart: change in jobs by industry since the internet

Each bar is the percentage change in the number of jobs in an industry between a starting census year and 2021. The dropdown changes the starting year; **1991** (the year the World Wide Web was invented) is the default. Hover for the numbers.

In [ ]:
END = 2021
STARTS = [1991, 1971, 2011]

rows = []
for start in STARTS:
    a = tidy[tidy["year"] == start].set_index("industry")
    b = tidy[tidy["year"] == END].set_index("industry")
    for industry in b.index:
        before, after = a.loc[industry, "jobs"], b.loc[industry, "jobs"]
        if before > 0:
            rows.append({"start": str(start), "industry": industry,
                         "jobs_start": before, "jobs_2021": after,
                         "change_pct": (after / before - 1) * 100})
change = pd.DataFrame(rows)
change["direction"] = change["change_pct"].map(lambda v: "More jobs" if v >= 0 else "Fewer jobs")
change[change["start"] == "1991"].sort_values("change_pct", ascending=False).round(1)

In [ ]:
NAVY, ACCENT, BLUE, MUTED, TEXT = "#1d3d63", "#c0341d", "#2f5ea8", "#4d6485", "#26406b"
FONT = "Helvetica Neue, Helvetica, Roboto, Arial, sans-serif"

pick = alt.param(name="start", value="1991",
                 bind=alt.binding_select(options=[str(s) for s in STARTS], name="Compare 2021 with "))

bars = alt.Chart(change).transform_filter(alt.datum.start == pick).mark_bar(cornerRadiusEnd=2).encode(
    x=alt.X("change_pct:Q", title="Change in number of jobs (%)"),
    y=alt.Y("industry:N", title=None, sort=alt.EncodingSortField(field="change_pct", order="descending"),
            axis=alt.Axis(labelLimit=260)),
    color=alt.Color("direction:N", title=None,
                    scale=alt.Scale(domain=["More jobs", "Fewer jobs"], range=[BLUE, ACCENT]),
                    legend=alt.Legend(orient="top")),
    tooltip=[alt.Tooltip("industry:N", title="Industry"),
             alt.Tooltip("start:N", title="From"),
             alt.Tooltip("jobs_start:Q", title="Jobs then", format=","),
             alt.Tooltip("jobs_2021:Q", title="Jobs in 2021", format=","),
             alt.Tooltip("change_pct:Q", title="Change (%)", format="+.0f")],
)
zero = alt.Chart(pd.DataFrame({"x": [0]})).mark_rule(color=MUTED).encode(x="x:Q")

cc5_scraper_chart = (bars + zero).add_params(pick).properties(
    width=520, height=420,
    title=alt.TitleParams("How jobs in Great Britain have changed since the internet",
                          subtitle=["Change in jobs by industry, census year selected below to 2021.",
                                    "Source: A Vision of Britain through Time (University of Portsmouth), scraped."],
                          anchor="start"),
).configure(font=FONT).configure_title(
    fontSize=17, color=NAVY, subtitleColor=MUTED, subtitleFontSize=12
).configure_axis(labelColor=TEXT, titleColor=TEXT, grid=False, domainColor=MUTED, tickColor=MUTED
).configure_view(stroke=None)

cc5_scraper_chart

## 6. Export the chart for the portfolio page

The tidy data is written **inside** the chart file, so the page needs only this one JSON file. (Alternatively, upload `uk_jobs_by_industry_tidy.csv` to GitHub and point the chart at its raw link.)

In [ ]:
cc5_scraper_chart.save("cc5_scraper_chart.json")
print("Saved cc5_scraper_chart.json")

# In Google Colab, download both files to your computer:
try:
    from google.colab import files
    files.download("uk_jobs_by_industry_tidy.csv")
    files.download("cc5_scraper_chart.json")
except ImportError:
    pass   # not running in Colab